# 05 — Enchantments & Elemental Damage (V1.5)

A deep dive into V1.5 features: weapon enchantments, elemental damage splitting,
elemental protection, reactive armor, and the per-hit metrics system.

**Sections:**
1. Spell strike enchantments (cast a spell on hit)
2. Slayer enchantments (bonus damage vs creature type)
3. Effect enchantments (life drain, mana drain, etc.)
4. Greater enchantments (Planar Fury, Void, Elemental Fury)
5. Elemental damage splitting and protection
6. Reactive armor
7. Inspecting per-hit metrics
8. DPS comparison: enchanted vs plain (V2)
9. Spell resistance impact on enchantments (V2)

In [ ]:
import logging
from pathlib import Path

from omega.config.enchantments import Enchantment
from omega.config.spells import Spell
from omega.model.constants import (
    SKILLID_ANATOMY, SKILLID_MACEFIGHTING, SKILLID_SPIRITSPEAK,
    SKILLID_SWORDSMANSHIP, SKILLID_TACTICS, SKILLID_WRESTLING,
)
from omega.shard import ShardData
from omega.simulation import (
    ArmorSpec, CombatantSpec, ParameterSweep, Scenario, Variable,
    WeaponSpec, run_scenario, run_sweep,
)
from omega.reporting.tables import comparison_table, summary_table, format_table_html
from omega.reporting.plots import (
    comparison_breakdown, comparison_overlay, damage_breakdown, damage_histogram,
    elemental_breakdown_chart, elemental_vs_parameter, enchantment_comparison,
)
from omega.logging import setup_logging
from IPython.display import HTML
import dataclasses

# Suppress noisy stub warnings — only show errors in notebook output
setup_logging(level=logging.ERROR)

# --- Shard setup ---
SHARD_ROOT = Path("submodules/zuluhotel_omega_2.5")
if not SHARD_ROOT.exists():
    SHARD_ROOT = Path("../submodules/zuluhotel_omega_2.5")
shard = ShardData.from_path(SHARD_ROOT)

RUN_KW = dict(shard=shard)

# --- Shared combatants ---
WEAPON = WeaponSpec(name="Broadsword", damage="3d6+2", attribute=SKILLID_SWORDSMANSHIP)
ATTACKER = CombatantSpec(
    name="Warrior",
    skills={SKILLID_SWORDSMANSHIP: 100, SKILLID_TACTICS: 100, SKILLID_ANATOMY: 100},
    str_=100, dex_=100, int_=25,
    class_levels={"IsWarrior": 5},
    weapon=WEAPON,
)
DEFENDER = CombatantSpec(
    name="Target Dummy", is_npc=True,
    str_=50, dex_=50, int_=50, hp=500,
    skills={SKILLID_WRESTLING: 60},  # NPC combat skill — affects hit chance
    armor=ArmorSpec(name="Plate", ar=30),
)
ITERATIONS = 200
BASE_SEED = 42

print("Setup complete.")

## 1. Spell Strike Enchantments

Spell strike weapons cast a spell on every hit (chance-based). The `enchant_with()`
API sets the hitscript and `HitWithSpell` property automatically.

We compare several spell tiers — from Circle 1 (Clumsy) to Circle 8 (Earthquake).

In [ ]:
# Build spell strike weapons across several spell circles.
# ChanceOfEffect (75%) and EffectCircle (10) are per-weapon properties that
# control how often the spell fires and at what power level.
BASE = WeaponSpec(
    name="Broadsword", damage="3d6+2", attribute=SKILLID_SWORDSMANSHIP,
    properties={"ChanceOfEffect": 75, "EffectCircle": 10},
)

spell_strikes = {
    "Plain (no enchant)": WEAPON,  # no ChanceOfEffect needed — no hitscript
    "C1: of Bungling":    BASE.enchant_with(Enchantment.OF_BUNGLING),      # Clumsy
    "C3: Daemon's Breath": BASE.enchant_with(Enchantment.OF_DAEMONS_BREATH), # Fireball
    "C5: of Disruption":  BASE.enchant_with(Enchantment.OF_DISRUPTION),    # Energy Bolt
    "C7: of Hellfire":    BASE.enchant_with(Enchantment.OF_HELLFIRE),      # Flame Strike
    "C8: Gaia's Wrath":   BASE.enchant_with(Enchantment.OF_GAIAS_WRATH),  # Earthquake
}

# Show the metadata for each
for label, w in spell_strikes.items():
    spell_id = w.properties.get("HitWithSpell", "—")
    chance = w.properties.get("ChanceOfEffect", "—")
    circle = w.properties.get("EffectCircle", "—")
    print(f"  {label:25s}  spell={str(spell_id):12s}  chance={chance}  circle={circle}")

In [ ]:
# Run each spell strike variant
spell_results = {}
for label, weapon in spell_strikes.items():
    atk = dataclasses.replace(ATTACKER, weapon=weapon)
    spell_results[label] = run_scenario(
        Scenario(attacker=atk, defender=DEFENDER, iterations=ITERATIONS, base_seed=BASE_SEED),
        **RUN_KW,
    )
    ds = spell_results[label].damage_stats
    r = spell_results[label].ratios
    rate = f"  strike_rate={r.spell_strike_rate:.0%}" if r.spell_strike_rate > 0 else ""
    print(f"  {label:25s}  mean={ds.mean:6.2f}{rate}")

In [ ]:
# Mean damage comparison across spell tiers
enchantment_comparison(spell_results, title="Spell Strike: Damage by Circle")

In [ ]:
# Full stat table with spell_strike_rate and elemental damage columns
rows = comparison_table(
    spell_results,
    stats=["mean", "mean_on_hit", "median", "min", "max", "p5", "p95", "hit_rate",
           "spell_strike_rate", "spell_strike_rate_on_hit", "elem_total_net"],
)
HTML(format_table_html(rows))

## 2. Slayer Enchantments

Slayer weapons deal bonus damage against a specific creature type. The slayer
script checks the defender's `Type` CProp — if it matches, damage is multiplied.

We test **Silver** (anti-Undead) against a matching Undead target vs a non-matching target.

In [ ]:
# Silver sword (anti-Undead slayer)
silver_sword = WEAPON.enchant_with(Enchantment.SILVER)
print(f"Silver sword: hitscript={silver_sword.hitscript}")
print(f"              SlayType={silver_sword.properties.get('SlayType')}")

# Two defenders: one Undead (matching), one generic (non-matching)
undead_target = dataclasses.replace(DEFENDER, name="Skeleton", properties={"Type": "Undead"})
generic_target = dataclasses.replace(DEFENDER, name="Bandit")

slayer_results = {}
for label, defender in [("vs Undead (match)", undead_target), ("vs Generic (no match)", generic_target)]:
    atk = dataclasses.replace(ATTACKER, weapon=silver_sword)
    slayer_results[label] = run_scenario(
        Scenario(attacker=atk, defender=defender, iterations=ITERATIONS, base_seed=BASE_SEED),
        **RUN_KW,
    )
    ds = slayer_results[label].damage_stats
    print(f"  {label:25s}  mean={ds.mean:6.2f}  max={ds.max:.0f}")

# Also run plain weapon as baseline
slayer_results["Plain (no slayer)"] = run_scenario(
    Scenario(attacker=ATTACKER, defender=undead_target, iterations=ITERATIONS, base_seed=BASE_SEED),
    **RUN_KW,
)
print(f"  {'Plain (no slayer)':25s}  mean={slayer_results['Plain (no slayer)'].damage_stats.mean:6.2f}")

In [ ]:
# Overlaid histograms — slayer match vs no-match vs plain
comparison_overlay(slayer_results, title="Silver Slayer: Matching vs Non-Matching Target")

In [ ]:
rows = comparison_table(
    slayer_results,
    stats=["mean", "median", "min", "max", "p5", "p95", "hit_rate"],
)
HTML(format_table_html(rows))

## 3. Effect Enchantments

Effect enchantments trigger special effects on hit: life drain, mana drain,
stamina drain, armor piercing, poison, banishing, and blinding. These appear
as side effects in the hit results rather than direct damage increases.

In [ ]:
# Compare all effect enchantments
effect_weapons = {
    "Plain":      WEAPON,
    "Piercing":   WEAPON.enchant_with(Enchantment.OF_PIERCING),
    "Bloody":     WEAPON.enchant_with(Enchantment.BLOODY),       # life drain
    "Vampiric":   WEAPON.enchant_with(Enchantment.VAMPIRIC),     # mana drain
    "Leech":      WEAPON.enchant_with(Enchantment.LEECH),        # stamina drain
    "Poisoned":   WEAPON.enchant_with(Enchantment.POISONED),
    "Blinding":   WEAPON.enchant_with(Enchantment.BLINDING),
}

effect_results = {}
for label, weapon in effect_weapons.items():
    atk = dataclasses.replace(ATTACKER, weapon=weapon)
    effect_results[label] = run_scenario(
        Scenario(attacker=atk, defender=DEFENDER, iterations=ITERATIONS, base_seed=BASE_SEED),
        **RUN_KW,
    )
    ds = effect_results[label].damage_stats
    r = effect_results[label].ratios
    extras = []
    if r.effect_rate > 0:
        extras.append(f"effect_rate={r.effect_rate:.0%}")
    if r.poison_rate > 0:
        extras.append(f"poison={r.poison_rate:.0%}")
    extra_str = f"  ({', '.join(extras)})" if extras else ""
    print(f"  {label:12s}  mean={ds.mean:6.2f}{extra_str}")

In [ ]:
enchantment_comparison(effect_results, title="Effect Enchantments: Damage Comparison")

In [ ]:
rows = comparison_table(
    effect_results,
    stats=["mean", "mean_on_hit", "p5", "p95", "hit_rate",
           "effect_rate", "effect_rate_on_hit",
           "drain_mean", "drain_mean_on_hit", "poison_rate"],
)
HTML(format_table_html(rows))

## 4. Greater Enchantments

The three greater enchantments are the most powerful weapon modifications:

- **Planar Fury** — deals holy + necro (planar) damage on hit
- **Void** — pure void damage burst
- **Elemental Fury** — triple-element (fire + air + earth) damage on hit

These add significant elemental damage on top of the physical base.

In [ ]:
# Greater enchantments vs plain
# Planar Fury and Elemental Fury gate on ChanceOfEffect (like spell strikes).
# Void has no chance gate — it always fires.
GREATER_BASE = WeaponSpec(
    name="Broadsword", damage="3d6+2", attribute=SKILLID_SWORDSMANSHIP,
    properties={"ChanceOfEffect": 75},
)

greater_weapons = {
    "Plain":          WEAPON,
    "Planar Fury":    GREATER_BASE.enchant_with(Enchantment.OF_PLANAR_FURY),
    "Void":           WEAPON.enchant_with(Enchantment.OF_THE_VOID),  # no ChanceOfEffect needed
    "Elemental Fury": GREATER_BASE.enchant_with(Enchantment.OF_ELEMENTAL_FURY),
}

greater_results = {}
for label, weapon in greater_weapons.items():
    atk = dataclasses.replace(ATTACKER, weapon=weapon)
    greater_results[label] = run_scenario(
        Scenario(attacker=atk, defender=DEFENDER, iterations=ITERATIONS, base_seed=BASE_SEED),
        **RUN_KW,
    )
    ds = greater_results[label].damage_stats
    r = greater_results[label].ratios
    eb = greater_results[label].elemental_breakdown
    dr = greater_results[label].drain_stats
    elem_str = ""
    if eb.total_net > 0:
        elem_parts = [f"{e}={d:.1f}" for e, d in eb.net_dict().items()]
        elem_str = f"  elem=[{', '.join(elem_parts)}]"
    drain_str = f"  drain_mean={dr.mean:.1f}" if dr.mean > 0 else ""
    rate = f"  effect_rate={r.effect_rate:.0%}" if r.effect_rate > 0 else ""
    print(f"  {label:16s}  mean={ds.mean:6.2f}{rate}{elem_str}{drain_str}")

In [ ]:
# Side-by-side damage bars showing physical + elemental split
enchantment_comparison(greater_results, title="Greater Enchantments: Physical + Elemental")

In [ ]:
# Elemental breakdown for Elemental Fury (fire + air + earth)
fury_cell = greater_results["Elemental Fury"]
elemental_breakdown_chart(fury_cell, title="Elemental Fury — Per-Element Breakdown")

In [ ]:
rows = comparison_table(
    greater_results,
    stats=["mean", "mean_on_hit", "median", "p5", "p95", "hit_rate",
           "effect_rate", "effect_rate_on_hit",
           "drain_mean", "drain_mean_on_hit", "elem_total_net", "elem_total_gross"],
)
HTML(format_table_html(rows))

## 5. Elemental Damage Splitting & Protection

Weapons with `ElementalDamage` split their physical damage across element types.
Defender protection CProps (`FireProtection`, `AirProtection`, etc.) reduce
elemental damage independently — physical portion ignores elemental protection.

This section sweeps defender fire protection from 0 to 100 on a 50/50 fire/physical weapon.

In [ ]:
# Fire weapon: 50% fire, 50% physical
fire_weapon = WeaponSpec(
    name="Flaming Broadsword", damage="3d6+2",
    attribute=SKILLID_SWORDSMANSHIP,
    properties={"ElementalDamage": "FIRE:50 PHYSICAL:50"},
)

# Sweep defender fire protection from 0 to 100
fire_sweep = ParameterSweep(
    scenario=Scenario(
        attacker=dataclasses.replace(ATTACKER, weapon=fire_weapon),
        defender=dataclasses.replace(DEFENDER, name="Protected Target"),
        iterations=100,
        base_seed=1234,
    ),
    variables=(
        Variable.from_range("defender", "properties.FireProtection", start=0, stop=100, step=10),
    ),
)

fire_result = run_sweep(fire_sweep, **RUN_KW)
print(f"{len(fire_result.cells)} cells completed in {fire_result.total_time:.1f}s")

In [ ]:
# Per-element damage curves — fire drops while physical stays flat
elemental_vs_parameter(
    fire_result,
    "defender.properties.FireProtection",
    title="Per-Element Damage vs Fire Protection",
)

In [ ]:
# Elemental breakdown at 0% vs 50% vs 100% fire protection
for prot_level in [0, 50, 100]:
    cell = fire_result.get_cell(**{"defender.properties.FireProtection": prot_level})
    if cell:
        eb = cell.elemental_breakdown
        ds = cell.damage_stats
        fire_net = eb.elements.get("fire")
        fire_str = f"fire_net={fire_net.net:.1f}" if fire_net else "fire=0"
        print(f"  FireProtection={prot_level:3d}%  total_mean={ds.mean:6.2f}  {fire_str}  elem_total={eb.total_net:.1f}")

In [ ]:
# Summary table across the sweep
rows = summary_table(
    fire_result,
    stats=["mean", "median", "p5", "p95", "hit_rate", "elem_total_net", "elem_total_gross"],
)
HTML(format_table_html(rows))

## 6. Reactive Armor

Reactive Armor reflects a portion of physical damage back to the attacker.
The `ReactiveArmor` CProp on the defender controls the power level:
`retaliation = CInt(basedamage * power / 100)`.

Against **player** attackers, the reflected damage is further divided by 8:
`actual = CInt(retaliation / 8)`.

**Important**: RA always triggers (100% rate), and it only affects the
**attacker** — the defender's incoming damage is unchanged. The stat to watch
is `absorbed_mean` for the reactive damage reflected back.

In [ ]:
# Defender with vs without reactive armor.
# Power=100 means full basedamage is reflected (then /8 vs player attackers).
# The attacker is a player, so actual reflected = CInt(CInt(basedmg * 100/100) / 8).
RA_POWER = 100
defender_plain = DEFENDER
defender_ra = dataclasses.replace(
    DEFENDER, name="Target (RA)", properties={"ReactiveArmor": RA_POWER},
)

ra_results = {}
for label, defender in [("No Reactive Armor", defender_plain), ("Reactive Armor", defender_ra)]:
    ra_results[label] = run_scenario(
        Scenario(attacker=ATTACKER, defender=defender, iterations=ITERATIONS, base_seed=BASE_SEED),
        **RUN_KW,
    )
    ds = ra_results[label].damage_stats
    r = ra_results[label].ratios
    rate = f"  reactive_rate={r.reactive_rate:.0%}" if r.reactive_rate > 0 else ""
    print(f"  {label:25s}  mean={ds.mean:6.2f}{rate}  RA_power={RA_POWER}")

In [ ]:
# Damage breakdown: base / absorbed / final (side by side)
# Note: RA reflects damage to the ATTACKER, so the defender's damage stats
# are identical with and without RA. The comparison_breakdown shows this clearly.
comparison_breakdown(ra_results, title="Reactive Armor: Damage Breakdown")

In [ ]:
rows = comparison_table(
    ra_results,
    stats=["mean", "mean_on_hit", "median", "p5", "p95", "hit_rate",
           "reactive_rate", "reactive_rate_on_hit", "absorbed_mean"],
)
HTML(format_table_html(rows))

## 7. Inspecting Per-Hit Metrics

Every `HitResult` carries a `metrics` dict populated by `__RecordSimulatorMetric()`
calls in the eScript code. This gives detailed per-hit visibility into the damage
pipeline — elemental splits, enchantment triggers, protection values, etc.

The `list:` prefix protocol appends structured data per hit (e.g., `list:elemental_applied`).

In [ ]:
# Run a single scenario with an Elemental Fury weapon and inspect raw hit data.
# Elemental Fury needs ChanceOfEffect to trigger its elemental damage.
fury_weapon = WeaponSpec(
    name="Broadsword", damage="3d6+2", attribute=SKILLID_SWORDSMANSHIP,
    properties={"ChanceOfEffect": 75},
).enchant_with(Enchantment.OF_ELEMENTAL_FURY)

fury_result = run_scenario(
    Scenario(
        attacker=dataclasses.replace(ATTACKER, weapon=fury_weapon),
        defender=DEFENDER,
        iterations=10,  # small batch for inspection
        base_seed=99,
    ),
    **RUN_KW,
)

# Inspect the first 3 hits
for i, hit in enumerate(fury_result.raw_results[:3]):
    print(f"--- Hit {i + 1} ---")
    print(f"  final_damage = {hit.final_damage:.2f}")
    print(f"  side_effects = {hit.side_effects}")
    print(f"  metrics keys = {sorted(hit.metrics.keys())}")
    # Show elemental/planar applied lists if present
    for metric_key in ("elemental_applied", "planar_applied"):
        entries = hit.metrics.get(metric_key, [])
        if entries:
            print(f"  {metric_key} ({len(entries)} entries):")
            for entry in entries:
                print(f"    {entry}")
    print()

In [ ]:
# Aggregate a custom metric across all hits — e.g., count how many hits
# triggered the enchantment effect vs plain physical-only hits.
# Greater enchantments use "planar_applied" while inline elemental uses "elemental_applied".
all_hits = fury_result.raw_results
enchant_triggered = sum(
    1 for h in all_hits
    if h.metrics.get("elemental_applied") or h.metrics.get("planar_applied")
)
print(f"Enchantment triggered: {enchant_triggered}/{len(all_hits)} hits "
      f"({enchant_triggered / len(all_hits):.0%})")

# Average elemental damage when the enchantment fires
elem_totals = []
for h in all_hits:
    total = 0.0
    for metric_key in ("elemental_applied", "planar_applied"):
        entries = h.metrics.get(metric_key, [])
        for e in entries:
            if isinstance(e, dict):
                total += e.get("dmg_net", 0)
    if total > 0:
        elem_totals.append(total)

if elem_totals:
    import statistics
    print(f"Mean elemental damage (when triggered): {statistics.mean(elem_totals):.2f}")
    print(f"Range: {min(elem_totals):.1f} – {max(elem_totals):.1f}")
else:
    print("No elemental damage recorded in hits.")

## 8. DPS Comparison: Enchanted vs Plain (V2)

Enchanted weapons add elemental/spell damage on top of physical damage per hit,
but weapon speed stays the same. This section shows DPS (not just per-hit damage)
to give the full picture of enchantment value.

In [ ]:
from omega.reporting.plots import dps_comparison

# Compare DPS across the key enchantment categories
dps_weapons = {
    "Plain":           WEAPON,
    "Daemon's Breath": BASE.enchant_with(Enchantment.OF_DAEMONS_BREATH),
    "Hellfire":        BASE.enchant_with(Enchantment.OF_HELLFIRE),
    "Elemental Fury":  GREATER_BASE.enchant_with(Enchantment.OF_ELEMENTAL_FURY),
    "Planar Fury":     GREATER_BASE.enchant_with(Enchantment.OF_PLANAR_FURY),
    "Void":            WEAPON.enchant_with(Enchantment.OF_THE_VOID),
}

dps_results = {}
for label, weapon in dps_weapons.items():
    atk = dataclasses.replace(ATTACKER, weapon=weapon)
    dps_results[label] = run_scenario(
        Scenario(attacker=atk, defender=DEFENDER, iterations=ITERATIONS, base_seed=BASE_SEED),
        **RUN_KW,
    )
    ts = dps_results[label].timing
    ds = dps_results[label].damage_stats
    dps_str = f"  effective_dps={ts.effective_dps:.2f}" if ts else ""
    print(f"  {label:18s}  mean={ds.mean:6.2f}{dps_str}")

dps_comparison(dps_results, title="DPS: Enchanted vs Plain")

In [ ]:
# DPS comparison table with timing and enchantment stats
rows = comparison_table(
    dps_results,
    stats=["mean", "mean_on_hit", "hit_rate", "swing_delay_ms", "effective_dps",
           "spell_strike_rate", "effect_rate", "elem_total_net"],
)
HTML(format_table_html(rows))

## 9. Spell Resistance Impact on Enchantments (V2)

Spell strike enchantments cast actual spells that go through the resistance check.
Defender class modifies resistance:
- **Mage**: +15 × ClasseBonusByLevel — strong resistance
- **Warrior**: -10, resist halved — weak against magic
- **Paladin**: +5 — moderate

This shows the same spell strike weapon against NPC (no class), Warrior, Mage,
and Paladin defenders to illustrate how spell resistance affects enchantment DPS.

In [ ]:
from omega.model.constants import SKILLID_MAGICRESISTANCE

# Hellfire spell strike weapon (high-circle for visible resistance effect)
RESIST_WEAPON = BASE.enchant_with(Enchantment.OF_HELLFIRE)

# Defenders: NPC (no class), then player classes with varying resistance
resist_defenders = {
    "NPC (no class)": DEFENDER,
    "Warrior": CombatantSpec(
        name="Warrior Defender",
        skills={SKILLID_SWORDSMANSHIP: 100, SKILLID_TACTICS: 100, SKILLID_ANATOMY: 100,
                SKILLID_WRESTLING: 80, SKILLID_MAGICRESISTANCE: 100},
        str_=120, dex_=80, int_=25, hp=500,
        class_levels={"IsWarrior": 5},
        armor=ArmorSpec(ar=30),
    ),
    "Mage": CombatantSpec(
        name="Mage Defender",
        skills={SKILLID_MAGICRESISTANCE: 100, SKILLID_WRESTLING: 80,
                SKILLID_MAGERY: 100, SKILLID_MEDITATION: 100},
        str_=50, dex_=50, int_=130, hp=500,
        class_levels={"IsMage": 5},
        armor=ArmorSpec(ar=30),
    ),
    "Paladin": CombatantSpec(
        name="Paladin Defender",
        skills={SKILLID_SWORDSMANSHIP: 100, SKILLID_TACTICS: 100,
                SKILLID_WRESTLING: 80, SKILLID_MAGICRESISTANCE: 100,
                SKILLID_MAGERY: 100},
        str_=110, dex_=80, int_=35, hp=500,
        class_levels={"IsPaladin": 5},
        armor=ArmorSpec(ar=30),
    ),
}

resist_atk = dataclasses.replace(ATTACKER, weapon=RESIST_WEAPON)
resist_results = {}
for label, defender in resist_defenders.items():
    resist_results[label] = run_scenario(
        Scenario(attacker=resist_atk, defender=defender,
                 iterations=ITERATIONS, base_seed=BASE_SEED),
        **RUN_KW,
    )
    ds = resist_results[label].damage_stats
    r = resist_results[label].ratios
    print(f"  vs {label:18s}  mean={ds.mean:6.2f}  "
          f"spell_strike={r.spell_strike_rate:.0%} (on hit: {r.spell_strike_rate_on_hit:.0%})")

In [ ]:
# Spell resistance impact — Warriors take more spell damage, Mages resist heavily
enchantment_comparison(resist_results, title="Hellfire Enchantment vs Defender Class")

In [ ]:
rows = comparison_table(
    resist_results,
    stats=["mean", "mean_on_hit", "median", "p5", "p95", "hit_rate",
           "spell_strike_rate", "spell_strike_rate_on_hit",
           "effective_dps", "elem_total_net"],
)
HTML(format_table_html(rows))